# Protocol v2 — Host-Graph GNN + Temporal Transformer (fair, multi-seed, shortcut-checked)
**Intelligent Adaptive IDS using Hybrid Transformer and Graph Neural Networks — NF-CSE-CIC-IDS2018-v3**

Kaggle: **GPU T4** → add the NF-CSE-CIC-IDS2018-v3 dataset → **Run All**.

What changed vs. the previous notebooks (supervisor's protocol):

| # | Change | Why |
|---|---|---|
| 1 | Full-file **Class × Day** counts, then **uniform sampling stratified by (Class, Day)** | "first 5000 rows" took all Benign from day 1 → the model could learn the day |
| 2 | Classes with < cap rows are taken **completely**; no SMOTE, no duplication | no fabricated flows |
| 3 | Train is balanced (Benign capped); **Val/Test keep a realistic Benign ratio (80%)** | real False-Alarm Rate |
| 4 | Three splits: **random**, **host-disjoint**, **temporal (per class)** | detects host memorisation and time shortcuts |
| 5 | Same split for all models; **5 seeds**; Wilcoxon / paired-t / McNemar / bootstrap | is Hybrid really better than Transformer? |
| 6 | 4 models on identical inputs: **GATv2, Transformer-only, Two-stage (frozen GNN), Hybrid (end-to-end)** | isolates graph vs. time vs. joint training |
| 7 | Metrics at **L1 (Benign/Attack), L2 (family), L3 (class), L3-merged (FTP+SlowHTTP)** + FAR / DR at a validation-chosen threshold | classes that the features cannot separate are reported honestly |
| 8 | **Ambiguity ceiling**: identical feature vectors with different labels | upper bound any model can reach |
| 9 | **Shortcut probes**: time-only model, structure-only GNN, TTL/TCP-window ablation, edge rewiring, window-grouping ablation | is the model using behaviour or identity? |
| 10 | **Per-class validation loss** | where the rising val-loss ("overfitting") comes from |

Results are written to disk after **every** run, so a Kaggle timeout does not lose finished work.

## 1. Environment

In [ ]:
import sys, subprocess, importlib.util
for pkg, mod in [("torch-geometric", "torch_geometric"), ("pyarrow", "pyarrow")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os, gc, re, json, time, math, random, shutil, zipfile, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, ttest_rel, binomtest

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score, matthews_corrcoef,
                             roc_auc_score, confusion_matrix)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv

warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))
elif not os.environ.get("IDS_ALLOW_CPU"):
    raise RuntimeError("No GPU. Kaggle: Settings -> Accelerator -> GPU T4, then Run All.")

## 2. Configuration
Full protocol = 3 splits × 5 seeds × 4 models (≈ 60 trainings). For a quick first check set `QUICK = True`.

In [ ]:
QUICK = False

SPLIT_SEED = 42                                   # the split is FIXED; seeds only change model training
SEEDS = [42, 43, 44, 45, 46]
SPLITS_TO_RUN = ["random", "host_disjoint", "temporal"]
MODELS = ["gatv2", "transformer", "two_stage", "hybrid"]

# ---- sampling ----
MAX_PER_CLASS = 5_000            # attack classes (per class, spread over its days)
BENIGN_POOL = 80_000             # Benign sampled (spread over all days); train uses only MAX_PER_CLASS*0.7 of it
EVAL_BENIGN_RATIO = 0.80         # Benign share in Val/Test (realistic). Train stays balanced.
CHUNK_SIZE = 500_000
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

# ---- features (locked list from the train-only MI ranking of the earlier runs) ----
LOCKED_FEATURES = [
    "IN_BYTES", "LONGEST_FLOW_PKT", "MAX_IP_PKT_LEN", "OUT_BYTES", "SRC_TO_DST_AVG_THROUGHPUT",
    "DST_TO_SRC_AVG_THROUGHPUT", "DURATION_IN", "FLOW_DURATION_MILLISECONDS", "CLIENT_TCP_FLAGS",
    "SRC_TO_DST_IAT_MAX", "SRC_TO_DST_IAT_AVG", "SRC_TO_DST_IAT_STDDEV", "TCP_FLAGS", "DURATION_OUT",
    "DST_TO_SRC_IAT_AVG", "DST_TO_SRC_IAT_MAX", "NUM_PKTS_UP_TO_128_BYTES", "OUT_PKTS",
    "DST_TO_SRC_IAT_STDDEV", "IN_PKTS", "L7_PROTO", "TCP_WIN_MAX_IN", "MAX_TTL", "MIN_TTL",
    "SERVER_TCP_FLAGS", "TCP_WIN_MAX_OUT", "NUM_PKTS_512_TO_1024_BYTES", "DST_TO_SRC_SECOND_BYTES",
    "SRC_TO_DST_SECOND_BYTES", "SHORTEST_FLOW_PKT", "MIN_IP_PKT_LEN", "NUM_PKTS_256_TO_512_BYTES",
]
FINGERPRINT_FEATURES = ["MIN_TTL", "MAX_TTL", "TCP_WIN_MAX_IN", "TCP_WIN_MAX_OUT"]   # host/OS fingerprints
LOG1P_FEATURES = True

# ---- windows ----
K = 32
WINDOW_GROUP = "src"             # "src" | "dst" | "global"

# ---- models ----
GNN_HIDDEN, GNN_HEADS, EMB_DIM, DROPOUT = 64, 2, 128, 0.20
D_MODEL, N_HEADS, N_LAYERS, FF_DIM = 128, 4, 2, 256

# ---- training ----
BATCH_SIZE = 512
LR = 1e-3                        # GATv2 / Transformer / Two-stage
HYBRID_GNN_LR = 1e-4             # warm-started GNN inside the Hybrid
HYBRID_FREEZE_EPOCHS = 3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.0            # protocol knob (e.g. 0.1) — keep identical for all models
DROPEDGE = 0.0                   # protocol knob (e.g. 0.15) — random message-edge dropout while training
MAX_EPOCHS = 150
PATIENCE = 20
MIN_EPOCHS = 10

# ---- evaluation ----
TARGET_FAR = 0.01                # threshold chosen on VAL so that Val FAR <= 1%
MERGE_GROUPS = {"FTP-BruteForce": "FTP/SlowHTTP", "DoS_attacks-SlowHTTPTest": "FTP/SlowHTTP"}
N_BOOTSTRAP = 1000

RUN_AMBIGUITY = True
RUN_TIME_PROBE = True
RUN_SHORTCUT_PROBES = True       # structure-only, fingerprint ablation, rewiring (random split, first seed)
RUN_GROUPING_ABLATION = True     # Hybrid with dst / global windows (random split, first seed)

if QUICK:
    SEEDS, SPLITS_TO_RUN = SEEDS[:1], ["random", "host_disjoint"]

# offline smoke-test overrides (not used on Kaggle)
_e = os.environ.get
MAX_EPOCHS = int(_e("IDS_MAX_EPOCHS", MAX_EPOCHS)); MIN_EPOCHS = min(MIN_EPOCHS, MAX_EPOCHS)
MAX_PER_CLASS = int(_e("IDS_MAX_PER_CLASS", MAX_PER_CLASS)); BENIGN_POOL = int(_e("IDS_BENIGN_POOL", BENIGN_POOL))
if _e("IDS_SEEDS"): SEEDS = [int(s) for s in _e("IDS_SEEDS").split(",")]
if _e("IDS_K"): K = int(_e("IDS_K"))

WORK_DIR = Path("/kaggle/working/PROTOCOL_V2") if Path("/kaggle/working").exists() else Path("./PROTOCOL_V2")
RES, FIG, MOD, PRED = (WORK_DIR / d for d in ["results", "figures", "models", "predictions"])
for p in (RES, FIG, MOD, PRED):
    p.mkdir(parents=True, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

json.dump({k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, dict, bool))},
          open(RES / "protocol_config.json", "w"), indent=2)
print("Output:", WORK_DIR, "| splits:", SPLITS_TO_RUN, "| seeds:", SEEDS)

## 3. Locate NF-CSE-CIC-IDS2018-v3 (CSV or ZIP)

In [ ]:
SRC_IP, DST_IP, SRC_PORT, DST_PORT = "IPV4_SRC_ADDR", "IPV4_DST_ADDR", "L4_SRC_PORT", "L4_DST_PORT"
T_START, T_END, PROTO, LABEL, BIN_LABEL = "FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS", "PROTOCOL", "Attack", "Label"
REQUIRED = [T_START, SRC_IP, DST_IP, LABEL]

def _has_schema(read_head):
    try:
        cols = [str(c).strip() for c in read_head().columns]
        return all(c in cols for c in REQUIRED)
    except Exception:
        return False

def find_source():
    roots = [Path(_e("IDS_DATA_DIR"))] if _e("IDS_DATA_DIR") else [Path("/kaggle/input")]
    for root in roots:
        for p in sorted(root.rglob("*.csv")):
            if "unsw" not in p.name.lower() and _has_schema(lambda: pd.read_csv(p, nrows=0)):
                return {"type": "csv", "path": p}
        for zp in sorted(root.rglob("*.zip")):
            try:
                with zipfile.ZipFile(zp) as z:
                    for m in z.namelist():
                        if m.lower().endswith(".csv") and "unsw" not in m.lower():
                            if _has_schema(lambda: pd.read_csv(z.open(m), nrows=0)):
                                return {"type": "zip", "path": zp, "member": m}
            except Exception:
                pass
    raise FileNotFoundError("NF-CSE-CIC-IDS2018-v3 CSV not found under /kaggle/input.")

def iter_chunks(src, usecols=None):
    kw = dict(chunksize=CHUNK_SIZE, low_memory=False, on_bad_lines="skip", usecols=usecols)
    if src["type"] == "csv":
        yield from pd.read_csv(src["path"], **kw)
    else:
        with zipfile.ZipFile(src["path"]) as z, z.open(src["member"]) as f:
            yield from pd.read_csv(f, **kw)

SOURCE = find_source()
print("Dataset:", SOURCE)

def day_of(ms):
    return pd.to_datetime(pd.to_numeric(ms, errors="coerce"), unit="ms", utc=True).dt.strftime("%Y-%m-%d")

## 4. Pass 1 — full-file Class × Day counts
This answers the team's question directly: **on which days does each class (including Benign) exist?**

In [ ]:
counts = defaultdict(int)
t0 = time.time()
for ch in iter_chunks(SOURCE, usecols=[T_START, LABEL]):
    lab = ch[LABEL].astype(str).str.strip()
    ok = ~lab.str.lower().isin(["", "nan", "none", "attack"])
    for (c, d), n in pd.DataFrame({"c": lab[ok], "d": day_of(ch.loc[ok, T_START])}).value_counts().items():
        counts[(c, d)] += int(n)
class_day = pd.Series(counts).unstack(fill_value=0).sort_index()
class_day.to_csv(RES / "full_dataset_class_by_day.csv")
print(f"Pass 1 done in {time.time() - t0:.0f}s — rows: {class_day.values.sum():,}")
display(class_day)
print("\nDays on which Benign exists:", int((class_day.loc["Benign"] > 0).sum()) if "Benign" in class_day.index else 0,
      "of", class_day.shape[1])

fig, ax = plt.subplots(figsize=(1.0 * class_day.shape[1] + 4, 0.45 * class_day.shape[0] + 1.5))
ax.imshow(np.log10(class_day.values + 1), cmap="Blues", aspect="auto")
ax.set_xticks(range(class_day.shape[1])); ax.set_xticklabels(class_day.columns, rotation=45, ha="right")
ax.set_yticks(range(class_day.shape[0])); ax.set_yticklabels(class_day.index)
for i in range(class_day.shape[0]):
    for j in range(class_day.shape[1]):
        v = class_day.values[i, j]
        if v:
            ax.text(j, i, f"{v:,}", ha="center", va="center", fontsize=7,
                    color="white" if np.log10(v + 1) > 0.6 * np.log10(class_day.values.max() + 1) else "black")
ax.set_title("Full dataset: flows per class per day"); plt.tight_layout()
plt.savefig(FIG / "01_class_by_day.png", dpi=160, bbox_inches="tight"); plt.show()

## 5. Pass 2 — uniform sampling stratified by (Class, Day)
Quota of a (class, day) cell = cap × (rows in that cell / rows of that class). Rows inside a cell are chosen
**uniformly at random over the whole file** (random key + keep the smallest keys). Classes smaller than the cap are kept whole.

In [ ]:
cls_tot = class_day.sum(1)
cap = {c: (BENIGN_POOL if c == "Benign" else MAX_PER_CLASS) for c in class_day.index}
quota = {}
for c in class_day.index:
    for d in class_day.columns:
        n = int(class_day.loc[c, d])
        if n:
            quota[(c, d)] = n if cls_tot[c] <= cap[c] else max(1, int(cap[c] * n / cls_tot[c]))

rng_s = np.random.default_rng(SPLIT_SEED)
kept = {}
t0 = time.time()
for ch in iter_chunks(SOURCE):
    ch.columns = [str(c).strip() for c in ch.columns]
    ch[LABEL] = ch[LABEL].astype(str).str.strip()
    ch = ch[~ch[LABEL].str.lower().isin(["", "nan", "none", "attack"])]
    ch = ch.assign(_day=day_of(ch[T_START]).values, _r=rng_s.random(len(ch)))
    for key, g in ch.groupby([LABEL, "_day"], sort=False):
        q = quota.get(key, 0)
        if q == 0:
            continue
        g = g if key not in kept else pd.concat([kept[key], g], ignore_index=True)
        kept[key] = g.nsmallest(q, "_r") if len(g) > q else g
    del ch; gc.collect()
df = pd.concat(kept.values(), ignore_index=True).drop(columns="_r")
del kept; gc.collect()
print(f"Pass 2 done in {time.time() - t0:.0f}s — sampled {len(df):,} rows")

# ---- cleaning (same structural rules as before) ----
df["_ts"] = pd.to_datetime(pd.to_numeric(df[T_START], errors="coerce"), unit="ms", utc=True)
n0 = len(df)
df = df.dropna(subset=[SRC_IP, DST_IP, "_ts", LABEL])
dkey = [c for c in [T_START, T_END, SRC_IP, DST_IP, SRC_PORT, DST_PORT, PROTO, LABEL] if c in df.columns]
n_dup = int(df.duplicated(subset=dkey).sum())
df = df.drop_duplicates(subset=dkey).sort_values("_ts", kind="mergesort").reset_index(drop=True)
df[SRC_IP] = df[SRC_IP].astype(str); df[DST_IP] = df[DST_IP].astype(str)

sample_class_day = df.groupby([LABEL, "_day"]).size().unstack(fill_value=0)
sample_class_day.to_csv(RES / "sample_class_by_day.csv")
audit = pd.DataFrame({"Item": ["Rows sampled", "Duplicates removed", "Rows used", "Classes", "Days",
                               "Unique src IPs", "Unique dst IPs"],
                      "Value": [n0, n_dup, len(df), df[LABEL].nunique(), df["_day"].nunique(),
                                df[SRC_IP].nunique(), df[DST_IP].nunique()]})
audit.to_csv(RES / "data_quality_audit.csv", index=False)
display(audit); display(sample_class_day)

CLASSES = sorted(df[LABEL].unique())
le = LabelEncoder().fit(CLASSES)
N_CLASSES = len(CLASSES)
BENIGN_ID = int(le.transform(["Benign"])[0])
Y_ALL = le.transform(df[LABEL])

## 6. Label hierarchy (L1 / L2 / L3 / L3-merged)
The model is always trained on the **15 original classes**. Coarser levels are computed from the same predictions.

In [ ]:
def family(name):
    n = name.lower()
    if n == "benign": return "Benign"
    if "ddos" in n: return "DDoS"
    if "dos" in n: return "DoS"
    if any(s in n for s in ["xss", "sql", "brute_force_-web", "web"]): return "Web"
    if "brute" in n: return "BruteForce"
    if "bot" in n: return "Bot"
    if "infil" in n: return "Infiltration"
    return "Other"

L2_NAMES = sorted({family(c) for c in CLASSES})
MAP_L2 = np.array([L2_NAMES.index(family(c)) for c in CLASSES])
L3M_NAMES = sorted({MERGE_GROUPS.get(c, c) for c in CLASSES})
MAP_L3M = np.array([L3M_NAMES.index(MERGE_GROUPS.get(c, c)) for c in CLASSES])
display(pd.DataFrame({"L3 class": CLASSES, "L2 family": [family(c) for c in CLASSES],
                      "L3-merged": [MERGE_GROUPS.get(c, c) for c in CLASSES]}))

## 7. Ambiguity ceiling — identical feature vectors with different labels
If two flows have exactly the same 32 feature values but different labels, **no model** can separate them.
This gives the upper bound for each class and identifies the pairs to report as merged.

In [ ]:
if RUN_AMBIGUITY:
    Fv = df[LOCKED_FEATURES].apply(pd.to_numeric, errors="coerce").round(6).astype(str).agg("|".join, axis=1)
    grp = pd.DataFrame({"f": Fv.values, "c": df[LABEL].values})
    lab_per_vec = grp.groupby("f")["c"].agg(lambda s: frozenset(s))
    conflict = grp["f"].map(lab_per_vec).apply(len) > 1
    grp["conflict"] = conflict.values
    best = grp.groupby(["f", "c"]).size().groupby(level=0).max().sum()
    print(f"Bayes-style ceiling on this sample (best possible L3 accuracy): {best / len(grp):.4f}")
    amb = grp.groupby("c")["conflict"].mean().rename("share_of_flows_with_conflicting_twin").sort_values(ascending=False)
    display(amb.to_frame())
    pair = defaultdict(int)
    for labs in lab_per_vec[lab_per_vec.apply(len) > 1].values:
        for a in labs:
            for b in labs:
                if a != b:
                    pair[(a, b)] += 1
    if pair:
        pm = pd.Series(pair).unstack(fill_value=0)
        pm.to_csv(RES / "ambiguity_conflicting_vectors_by_class_pair.csv")
        print("Distinct feature vectors shared between class pairs:"); display(pm)
    amb.to_csv(RES / "ambiguity_per_class.csv")

## 8. The three splits
* **random** — stratified by (class, day). Comparable with the literature; hosts are shared between train and test.
* **host_disjoint** — every source IP belongs to exactly one split. Classes generated by fewer than 3 source IPs
  (or whose IPs cannot be balanced) **cannot** be host-disjoint; for those classes the rows are split in time blocks
  instead, and the report says so explicitly.
* **temporal** — inside every class: first 70% of its flows (by time) → train, next 15% → val, last 15% → test.

After splitting: train Benign is capped to a balanced size; Val/Test keep `EVAL_BENIGN_RATIO` Benign.

In [ ]:
def _safe_strat(labels, fallback):
    vc = labels.value_counts()
    return labels.where(labels.map(vc) >= 10, fallback)

def split_random():
    strat = _safe_strat(df[LABEL] + "|" + df["_day"], df[LABEL])
    idx = np.arange(len(df))
    tr, tmp = train_test_split(idx, test_size=VAL_FRAC + TEST_FRAC, random_state=SPLIT_SEED, stratify=strat)
    s2 = _safe_strat(strat.iloc[tmp].reset_index(drop=True), df[LABEL].iloc[tmp].reset_index(drop=True))
    va, te = train_test_split(tmp, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), random_state=SPLIT_SEED, stratify=s2)
    rep = pd.DataFrame({"class": CLASSES, "mode": "random (class×day stratified)"})
    return tr, va, te, rep

def _time_blocks(ix):
    ix = df.loc[ix].sort_values("_ts", kind="mergesort").index.values
    a, b = int(len(ix) * TRAIN_FRAC), int(len(ix) * (TRAIN_FRAC + VAL_FRAC))
    return ix[:a], ix[a:b], ix[b:]

def split_temporal():
    parts = [_time_blocks(g.index.values) for _, g in df.groupby(LABEL)]
    tr, va, te = (np.concatenate([p[i] for p in parts]) for i in range(3))
    rep = pd.DataFrame({"class": CLASSES, "mode": "temporal blocks per class"})
    return tr, va, te, rep

def split_host_disjoint():
    rng = np.random.default_rng(SPLIT_SEED)
    target = np.array([TRAIN_FRAC, VAL_FRAC, TEST_FRAC])
    assign = {}
    for c in df.groupby(LABEL)[SRC_IP].nunique().sort_values().index:       # fewest IPs first
        ipc = df.loc[df[LABEL] == c, SRC_IP].value_counts()
        if len(ipc) < 3:
            continue
        load = np.zeros(3)
        for ip, n in ipc.items():
            if ip in assign:
                load[assign[ip]] += n
        free = [ip for ip in ipc.index if ip not in assign]
        rng.shuffle(free)
        for ip in sorted(free, key=lambda x: -ipc[x]):                       # big IPs first, then fill deficits
            s = int(np.argmax(target * ipc.sum() - load)); assign[ip] = s; load[s] += ipc[ip]
    row_split = df[SRC_IP].map(assign)
    out, rep = ([], [], []), []
    for c, g in df.groupby(LABEL):
        s = row_split.loc[g.index]
        shares = np.array([(s == k).mean() for k in range(3)])
        n_ip = g[SRC_IP].nunique()
        if s.isna().any() or shares[0] < 0.3 or shares[1] < 0.05 or shares[2] < 0.05:
            blocks, mode = _time_blocks(g.index.values), f"FALLBACK time blocks ({n_ip} src IPs) — host overlap allowed"
        else:
            blocks, mode = tuple(g.index.values[(s == k).values] for k in range(3)), f"host-disjoint ({n_ip} src IPs)"
        for k in range(3):
            out[k].append(blocks[k])
        rep.append({"class": c, "mode": mode, "src_ips": n_ip})
    tr, va, te = (np.concatenate(o) for o in out)
    return tr, va, te, pd.DataFrame(rep)

SPLITTERS = {"random": split_random, "host_disjoint": split_host_disjoint, "temporal": split_temporal}

def rebalance(tr, va, te):
    rng = np.random.default_rng(SPLIT_SEED)
    def cap_benign(ix, n_keep):
        ix = np.asarray(ix); b = ix[Y_ALL[ix] == BENIGN_ID]; a = ix[Y_ALL[ix] != BENIGN_ID]
        if len(b) > n_keep:
            b = rng.choice(b, n_keep, replace=False)
        return np.sort(np.concatenate([a, b]))          # df is time-sorted -> sorted index = chronological
    tr = cap_benign(tr, int(MAX_PER_CLASS * TRAIN_FRAC))
    ev = lambda ix: cap_benign(ix, int((Y_ALL[ix] != BENIGN_ID).sum() * EVAL_BENIGN_RATIO / (1 - EVAL_BENIGN_RATIO)))
    return tr, ev(va), ev(te)

## 9. Train-only preprocessing, host graph, temporal windows

In [ ]:
def to_num(part, cols):
    X = part[cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return np.sign(X) * np.log1p(np.abs(X)) if LOG1P_FEATURES else X

def build_graph(part, X):
    src, dst = part[SRC_IP].values, part[DST_IP].values
    hosts, inv = np.unique(np.concatenate([src, dst]), return_inverse=True)
    s, d = inv[:len(src)], inv[len(src):]
    n = len(hosts)
    out_d, in_d = np.bincount(s, minlength=n), np.bincount(d, minlength=n)
    u_out = pd.Series(d).groupby(s).nunique().reindex(range(n), fill_value=0).values
    u_in = pd.Series(s).groupby(d).nunique().reindex(range(n), fill_value=0).values
    nx_ = np.log1p(np.stack([out_d, in_d, out_d + in_d, u_out, u_in], 1).astype(np.float32))
    nx_ = (nx_ - nx_.mean(0)) / (nx_.std(0) + 1e-6)
    ei = np.stack([np.concatenate([s, d]), np.concatenate([d, s])])
    return Data(x=torch.tensor(nx_, dtype=torch.float32), edge_index=torch.tensor(ei, dtype=torch.long),
                edge_attr=torch.tensor(np.concatenate([X, X])), flow_src=torch.tensor(s, dtype=torch.long),
                flow_dst=torch.tensor(d, dtype=torch.long), flow_attr=torch.tensor(X)).to(device)

def build_windows(part, k, group):
    """Row i = [k-1 previous flows of the same host ..., flow i]; -1 = left padding. Past only."""
    n = len(part)
    t = (part["_ts"].values.astype("datetime64[ns]").astype(np.int64) / 1e9)
    g = np.zeros(n, dtype=np.int64) if group == "global" else \
        pd.factorize(part[SRC_IP if group == "src" else DST_IP])[0]
    order = np.lexsort((np.arange(n), t, g))
    gs = g[order]
    pos = np.arange(n) - np.searchsorted(gs, gs, side="left")
    seq = np.full((n, k), -1, dtype=np.int64)
    r = np.arange(n)
    for j in range(k):
        lag = k - 1 - j
        ok = pos >= lag
        seq[order[ok], j] = order[r[ok] - lag]
    valid = seq >= 0
    dt = np.zeros((n, k), dtype=np.float32)
    tgt = np.broadcast_to(t[:, None], (n, k))
    dt[valid] = np.log1p(np.maximum(tgt[valid] - t[seq[valid]], 0))
    return seq, dt

def prepare(idx3, features, group=WINDOW_GROUP, k=K):
    parts = {n: df.loc[ix].reset_index(drop=True) for n, ix in zip(["train", "val", "test"], idx3)}
    Xtr = to_num(parts["train"], features)
    imp = SimpleImputer(strategy="median").fit(Xtr)
    sc = StandardScaler().fit(imp.transform(Xtr))
    S = {"features": features, "imputer": imp, "scaler": sc, "group": group, "k": k}
    for n, part in parts.items():
        X = np.clip(sc.transform(imp.transform(to_num(part, features))), -10, 10).astype(np.float32)
        seq, dt = build_windows(part, k, group)
        S[n] = {"df": part, "n": len(part), "graph": build_graph(part, X), "X": torch.tensor(X, device=device),
                "y": torch.tensor(le.transform(part[LABEL]), dtype=torch.long, device=device),
                "seq": torch.tensor(seq, device=device), "dt": torch.tensor(dt, device=device)}
    return S

## 10. Models — identical building blocks for all four
| model | sees graph | sees time |
|---|---|---|
| `gatv2` | ✔ | ✘ |
| `transformer` | ✘ (raw flow features as tokens) | ✔ |
| `two_stage` | ✔ frozen GATv2 embeddings | ✔ |
| `hybrid` | ✔ GATv2 trained jointly (warm-started, frozen for the first epochs) | ✔ |

In [ ]:
class GATEncoder(nn.Module):
    def __init__(self, node_in, edge_in):
        super().__init__()
        self.gat1 = GATv2Conv(node_in, GNN_HIDDEN, heads=GNN_HEADS, concat=True, edge_dim=edge_in, dropout=DROPOUT)
        self.gat2 = GATv2Conv(GNN_HIDDEN * GNN_HEADS, GNN_HIDDEN, heads=1, concat=False, edge_dim=edge_in, dropout=DROPOUT)
        self.drop = nn.Dropout(DROPOUT)
        self.edge_proj = nn.Sequential(nn.Linear(2 * GNN_HIDDEN + edge_in, EMB_DIM), nn.ReLU(), nn.Dropout(DROPOUT))

    def node_embed(self, g):
        ei, ea = g.edge_index, g.edge_attr
        if self.training and DROPEDGE > 0:
            keep = torch.rand(ei.size(1), device=ei.device) >= DROPEDGE
            ei, ea = ei[:, keep], ea[keep]
        x = self.drop(F.elu(self.gat1(g.x, ei, ea)))
        return F.elu(self.gat2(x, ei, ea))

    def flow_embed(self, z, g, idx):
        return self.edge_proj(torch.cat([z[g.flow_src[idx]], z[g.flow_dst[idx]], g.flow_attr[idx]], 1))


class TemporalTransformer(nn.Module):
    def __init__(self, tok_in, k):
        super().__init__()
        self.inp = nn.Linear(tok_in + 1, D_MODEL)                      # +1 = log time-gap channel
        self.pos = nn.Parameter(torch.randn(1, k, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, FF_DIM, DROPOUT, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, N_LAYERS, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(D_MODEL)

    def forward(self, tok, dt, pad):
        h = self.inp(torch.cat([tok, dt.unsqueeze(-1)], -1)) + self.pos[:, -tok.size(1):]
        return self.norm(self.enc(h, src_key_padding_mask=pad)[:, -1])


class IDSModel(nn.Module):
    def __init__(self, kind, node_in, edge_in, k):
        super().__init__()
        self.kind = kind
        if kind in ("gatv2", "hybrid"):
            self.gnn = GATEncoder(node_in, edge_in)
        if kind in ("transformer", "two_stage", "hybrid"):
            self.temporal = TemporalTransformer(edge_in if kind == "transformer" else EMB_DIM, k)
        rep = {"gatv2": EMB_DIM, "transformer": D_MODEL, "two_stage": D_MODEL, "hybrid": D_MODEL + EMB_DIM}[kind]
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(rep, N_CLASSES))

    def context(self, P):
        return self.gnn.node_embed(P["graph"]) if hasattr(self, "gnn") else None

    def represent(self, P, idx, ctx):
        if self.kind == "gatv2":
            return self.gnn.flow_embed(ctx, P["graph"], idx)
        seq = P["seq"][idx]; pad = seq < 0; safe = seq.clamp(min=0)
        if self.kind == "transformer":
            tok = P["X"][safe]
        elif self.kind == "two_stage":
            tok = P["emb"][safe]
        else:
            tok = self.gnn.flow_embed(ctx, P["graph"], safe.reshape(-1)).view(*safe.shape, -1)
        tok = tok.masked_fill(pad.unsqueeze(-1), 0.0)
        h = self.temporal(tok, P["dt"][idx], pad)
        return torch.cat([h, tok[:, -1]], 1) if self.kind == "hybrid" else h

    def forward(self, P, idx, ctx):
        return self.head(self.represent(P, idx, ctx))

## 11. Training (validation-only selection) and evaluation (L1 / L2 / L3 / L3-merged, FAR, DR)

In [ ]:
@torch.no_grad()
def predict(model, P, want_repr=False, bs=4096):
    model.eval()
    ctx = model.context(P)
    L, R = [], []
    for b in range(0, P["n"], bs):
        idx = torch.arange(b, min(b + bs, P["n"]), device=device)
        r = model.represent(P, idx, ctx)
        L.append(model.head(r)); R.append(r)
    return (torch.cat(L), torch.cat(R)) if want_repr else torch.cat(L)

def train_model(kind, S, seed, init_gnn=None):
    set_seed(seed)
    tr = S["train"]
    model = IDSModel(kind, tr["graph"].x.shape[1], tr["X"].shape[1], S["k"]).to(device)
    groups = [{"params": [p for n, p in model.named_parameters() if not n.startswith("gnn.")], "lr": LR}]
    if hasattr(model, "gnn"):
        if init_gnn is not None:
            model.gnn.load_state_dict(init_gnn)
        groups.append({"params": model.gnn.parameters(), "lr": HYBRID_GNN_LR if kind == "hybrid" else LR})
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=8)
    w = torch.tensor(compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=tr["y"].cpu().numpy()),
                     dtype=torch.float32, device=device)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTHING)
    freeze = HYBRID_FREEZE_EPOCHS if (kind == "hybrid" and init_gnn is not None) else 0
    best, best_state, best_ep, bad, hist = -1.0, None, 0, 0, []
    yv = S["val"]["y"].cpu().numpy()
    for ep in range(1, MAX_EPOCHS + 1):
        if hasattr(model, "gnn"):
            for p in model.gnn.parameters():
                p.requires_grad_(ep > freeze)
        model.train()
        if ep <= freeze:
            model.gnn.eval()
        perm = torch.randperm(tr["n"], device=device); tot = 0.0
        for b in range(0, tr["n"], BATCH_SIZE):
            idx = perm[b:b + BATCH_SIZE]
            loss = crit(model(tr, idx, model.context(tr)), tr["y"][idx])
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.item() * len(idx)
        vl = predict(model, S["val"])
        vf1 = f1_score(yv, vl.argmax(1).cpu().numpy(), average="macro", zero_division=0)
        sched.step(vf1)
        hist.append({"epoch": ep, "train_loss": tot / tr["n"], "val_loss": crit(vl, S["val"]["y"]).item(), "val_macro_f1": vf1})
        if vf1 > best + 1e-5:
            best, best_ep, bad = vf1, ep, 0
            best_state = {k_: v.detach().clone() for k_, v in model.state_dict().items()}
        else:
            bad += 1
        if ep >= MIN_EPOCHS and bad >= PATIENCE:
            break
    model.load_state_dict(best_state)
    return model, {"best_epoch": best_ep, "stopped_epoch": ep, "best_val_macro_f1": best}, pd.DataFrame(hist)

def attack_threshold(val_logits, yv):
    score = 1 - torch.softmax(val_logits, 1)[:, BENIGN_ID].cpu().numpy()
    b = score[yv == BENIGN_ID]
    return float(np.quantile(b, 1 - TARGET_FAR)) if len(b) else 0.5

def evaluate(test_logits, yt, thr):
    prob = torch.softmax(test_logits, 1).cpu().numpy(); p = prob.argmax(1)
    score = 1 - prob[:, BENIGN_ID]
    is_att = yt != BENIGN_ID
    m = {"L3 Macro-F1": f1_score(yt, p, average="macro", zero_division=0),
         "L3 MCC": matthews_corrcoef(yt, p), "L3 Balanced Acc": balanced_accuracy_score(yt, p),
         "L3-merged Macro-F1": f1_score(MAP_L3M[yt], MAP_L3M[p], average="macro", zero_division=0),
         "L2 Macro-F1": f1_score(MAP_L2[yt], MAP_L2[p], average="macro", zero_division=0),
         "L1 FAR (argmax)": float((p[~is_att] != BENIGN_ID).mean()) if (~is_att).any() else np.nan,
         "L1 DR (argmax)": float((p[is_att] != BENIGN_ID).mean()),
         f"L1 FAR @val-thr": float((score[~is_att] > thr).mean()) if (~is_att).any() else np.nan,
         f"L1 DR @val-thr": float((score[is_att] > thr).mean()),
         "L1 ROC-AUC": roc_auc_score(is_att, score) if 0 < is_att.mean() < 1 else np.nan}
    pc = f1_score(yt, p, labels=np.arange(N_CLASSES), average=None, zero_division=0)
    return m, pc, p, score

def per_class_val_loss(val_logits, yv):
    l = F.cross_entropy(val_logits, torch.tensor(yv, device=val_logits.device), reduction="none").cpu().numpy()
    return pd.Series(l).groupby(yv).mean().reindex(range(N_CLASSES)).values

RESULTS, PERCLASS, VALLOSS, SPLIT_REPORTS = [], [], [], []
safe = lambda t: re.sub(r"[^A-Za-z0-9_.-]+", "_", t)

def run_and_log(kind, S, seed, split, tag=None, init_gnn=None, extra=None):
    tag = tag or kind
    t0 = time.time()
    model, info, hist = train_model(kind, S, seed, init_gnn)
    vl, tl = predict(model, S["val"]), predict(model, S["test"])
    yv, yt = S["val"]["y"].cpu().numpy(), S["test"]["y"].cpu().numpy()
    thr = attack_threshold(vl, yv)
    m, pc, p, score = evaluate(tl, yt, thr)
    row = {"split": split, "model": tag, "seed": seed, **m, **info, "seconds": round(time.time() - t0), **(extra or {})}
    RESULTS.append(row)
    PERCLASS.append({"split": split, "model": tag, "seed": seed, **dict(zip(CLASSES, pc))})
    VALLOSS.append({"split": split, "model": tag, "seed": seed, **dict(zip(CLASSES, per_class_val_loss(vl, yv)))})
    np.savez_compressed(PRED / f"{split}__{safe(tag)}__seed{seed}.npz", y=yt, pred=p, score=score)
    hist.to_csv(RES / f"history__{split}__{safe(tag)}__seed{seed}.csv", index=False)
    pd.DataFrame(RESULTS).to_csv(RES / "all_runs.csv", index=False)               # saved after every run
    pd.DataFrame(PERCLASS).to_csv(RES / "per_class_f1_all_runs.csv", index=False)
    pd.DataFrame(VALLOSS).to_csv(RES / "per_class_val_loss_all_runs.csv", index=False)
    print(f"  {split:13s} {tag:28s} seed={seed}  L3-F1={m['L3 Macro-F1']:.4f}  L3m-F1={m['L3-merged Macro-F1']:.4f}  "
          f"FAR@thr={m['L1 FAR @val-thr']:.4f}  DR@thr={m['L1 DR @val-thr']:.4f}  "
          f"(best ep {info['best_epoch']}/{info['stopped_epoch']}, {row['seconds']}s)")
    return model

## 12. Main experiment — every split × every seed × every model (same split for all models)

In [ ]:
def zero_features(S):
    Z = dict(S)
    for n in ["train", "val", "test"]:
        P = dict(S[n]); g = P["graph"].clone()
        g.flow_attr = torch.zeros_like(g.flow_attr); g.edge_attr = torch.zeros_like(g.edge_attr)
        P["graph"], P["X"] = g, torch.zeros_like(P["X"]); Z[n] = P
    return Z

def rewire_test(S, seed):
    """Same flows/features, but every flow's destination host is shuffled -> graph structure destroyed."""
    R = dict(S); P = dict(S["test"]); part = P["df"].copy()
    part[DST_IP] = np.random.default_rng(seed).permutation(part[DST_IP].values)
    X = P["X"].cpu().numpy()
    P["graph"] = build_graph(part, X)
    seq, dt = build_windows(part, S["k"], S["group"])
    P["seq"], P["dt"] = torch.tensor(seq, device=device), torch.tensor(dt, device=device)
    R["test"] = P
    return R

def set_embeddings(gat, S):
    for n in ["train", "val", "test"]:
        S[n]["emb"] = predict(gat, S[n], want_repr=True)[1]

for split in SPLITS_TO_RUN:
    print(f"\n==================== SPLIT: {split} ====================")
    tr, va, te, rep = SPLITTERS[split]()
    tr, va, te = rebalance(tr, va, te)
    sizes = pd.DataFrame({n: pd.Series(Y_ALL[ix]).map(dict(enumerate(CLASSES))).value_counts()
                          for n, ix in [("train", tr), ("val", va), ("test", te)]}).fillna(0).astype(int)
    rep = rep.set_index("class").join(sizes)
    tr_hosts = set(df.loc[tr, SRC_IP])
    overlap = float(df.loc[te, SRC_IP].isin(tr_hosts).mean())
    rep.to_csv(RES / f"split_report__{split}.csv")
    SPLIT_REPORTS.append({"split": split, "test_rows_whose_src_ip_is_in_train": overlap,
                          "benign_share_val": float((Y_ALL[va] == BENIGN_ID).mean()),
                          "benign_share_test": float((Y_ALL[te] == BENIGN_ID).mean())})
    print(f"Test rows whose source host also appears in train: {overlap:.1%}")
    if SPLIT_REPORTS[-1]["benign_share_test"] < EVAL_BENIGN_RATIO - 0.05:
        print(f"WARNING: test Benign share is {SPLIT_REPORTS[-1]['benign_share_test']:.0%} < target "
              f"{EVAL_BENIGN_RATIO:.0%} -> increase BENIGN_POOL.")
    display(rep)

    S = prepare((tr, va, te), LOCKED_FEATURES)

    if split == "random" and RUN_TIME_PROBE:
        # Time-only probe: if the capture day/hour alone predicts the class, time is a shortcut.
        def tfeat(P):
            t = P["df"]["_ts"]; t0_ = df["_ts"].min()
            return np.stack([(t - t0_).dt.days, t.dt.hour, t.dt.hour * 60 + t.dt.minute], 1)
        rf = RandomForestClassifier(300, n_jobs=-1, random_state=SPLIT_SEED, class_weight="balanced")
        rf.fit(tfeat(S["train"]), S["train"]["y"].cpu().numpy())
        yt = S["test"]["y"].cpu().numpy(); pt = rf.predict(tfeat(S["test"]))
        RESULTS.append({"split": split, "model": "probe: time-only RF (day, hour, minute)", "seed": SPLIT_SEED,
                        "L3 Macro-F1": f1_score(yt, pt, average="macro"),
                        "L1 FAR (argmax)": float((pt[yt == BENIGN_ID] != BENIGN_ID).mean()),
                        "L1 DR (argmax)": float((pt[yt != BENIGN_ID] != BENIGN_ID).mean())})
        print(f"  time-only probe: L3 Macro-F1 = {RESULTS[-1]['L3 Macro-F1']:.4f}  (high = timestamp shortcut exists)")

    keep = {}
    for si, seed in enumerate(SEEDS):
        print(f"  --- seed {seed} ---")
        gat = run_and_log("gatv2", S, seed, split)
        gat_state = {k_[4:]: v.clone() for k_, v in gat.state_dict().items() if k_.startswith("gnn.")}
        if "two_stage" in MODELS:
            set_embeddings(gat, S)
            run_and_log("two_stage", S, seed, split)
        if "transformer" in MODELS:
            run_and_log("transformer", S, seed, split)
        hyb = run_and_log("hybrid", S, seed, split, init_gnn=gat_state) if "hybrid" in MODELS else None
        if si == 0:
            keep = {"gat": gat, "hyb": hyb, "gat_state": gat_state}
            torch.save(gat.state_dict(), MOD / f"gatv2__{split}__seed{seed}.pt")
            if hyb is not None:
                torch.save(hyb.state_dict(), MOD / f"hybrid__{split}__seed{seed}.pt")
        else:
            del gat, hyb
        gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

    # ---------------- shortcut probes (first seed only) ----------------
    if split == "random" and RUN_SHORTCUT_PROBES:
        s0 = SEEDS[0]
        print("  --- shortcut probes ---")
        run_and_log("gatv2", zero_features(S), s0, split, tag="probe: structure-only GATv2 (no flow features)")
        nofp = [f for f in LOCKED_FEATURES if f not in FINGERPRINT_FEATURES]
        S_fp = prepare((tr, va, te), nofp)
        g_fp = run_and_log("gatv2", S_fp, s0, split, tag="probe: GATv2 without TTL/TCP-window")
        run_and_log("hybrid", S_fp, s0, split, tag="probe: Hybrid without TTL/TCP-window",
                    init_gnn={k_[4:]: v for k_, v in g_fp.state_dict().items() if k_.startswith("gnn.")})
        del S_fp, g_fp; gc.collect()
        R = rewire_test(S, s0)
        for name, mdl in [("GATv2", keep["gat"]), ("Hybrid", keep["hyb"])]:
            if mdl is None:
                continue
            yv, yt = S["val"]["y"].cpu().numpy(), R["test"]["y"].cpu().numpy()
            m, pc, _, _ = evaluate(predict(mdl, R["test"]), yt, attack_threshold(predict(mdl, S["val"]), yv))
            RESULTS.append({"split": split, "model": f"probe: {name} on rewired test graph", "seed": s0, **m})
            print(f"  rewired test graph — {name}: L3-F1={m['L3 Macro-F1']:.4f}")
        del R

    if split == "random" and RUN_GROUPING_ABLATION and keep.get("hyb") is not None:
        for grp_ in ["dst", "global"]:
            S_g = prepare((tr, va, te), LOCKED_FEATURES, group=grp_)
            run_and_log("hybrid", S_g, SEEDS[0], split, tag=f"ablation: Hybrid windows by {grp_}",
                        init_gnn=keep["gat_state"])
            del S_g; gc.collect()

    del S, keep; gc.collect()
    pd.DataFrame(RESULTS).to_csv(RES / "all_runs.csv", index=False)

pd.DataFrame(SPLIT_REPORTS).to_csv(RES / "split_overview.csv", index=False)
display(pd.DataFrame(SPLIT_REPORTS))

## 13. Summary — mean ± std over seeds

In [ ]:
res = pd.DataFrame(RESULTS)
KEY = ["L3 Macro-F1", "L3-merged Macro-F1", "L2 Macro-F1", "L3 MCC", "L1 FAR @val-thr", "L1 DR @val-thr",
       "L1 FAR (argmax)", "L1 DR (argmax)", "L1 ROC-AUC", "best_epoch"]
main = res[res["model"].isin(MODELS)]
summary = main.groupby(["split", "model"], sort=False)[KEY].agg(["mean", "std"]).round(4)
summary.to_csv(RES / "summary_mean_std.csv")
display(summary)
probes = res[~res["model"].isin(MODELS)]
if len(probes):
    probes.to_csv(RES / "probes_and_ablations.csv", index=False)
    display(probes[["split", "model", "L3 Macro-F1", "L3-merged Macro-F1", "L1 FAR (argmax)", "L1 DR (argmax)"]].round(4))

## 14. Is the difference real? — paired tests over seeds, McNemar per seed, bootstrap CI
* **paired t-test / Wilcoxon** on per-seed L3 Macro-F1 (note: with 5 seeds the smallest possible two-sided Wilcoxon p is 0.0625).
* **McNemar** (exact) on the same test flows for each seed: are the two models' errors different?
* **Bootstrap 95% CI** of the Macro-F1 difference on the first seed's test set.

In [ ]:
def load_pred(split, model, seed):
    z = np.load(PRED / f"{split}__{model}__seed{seed}.npz"); return z["y"], z["pred"]

PAIRS = [("hybrid", "transformer"), ("hybrid", "gatv2"), ("hybrid", "two_stage"), ("two_stage", "transformer"),
         ("transformer", "gatv2")]
rows = []
for split in SPLITS_TO_RUN:
    for a, b in PAIRS:
        if a not in MODELS or b not in MODELS:
            continue
        A = main[(main.split == split) & (main.model == a)].sort_values("seed")["L3 Macro-F1"].values
        B = main[(main.split == split) & (main.model == b)].sort_values("seed")["L3 Macro-F1"].values
        if len(A) != len(B) or len(A) == 0:
            continue
        d = A - B
        p_t = ttest_rel(A, B).pvalue if len(A) > 1 else np.nan
        p_w = wilcoxon(A, B).pvalue if len(A) > 1 and np.any(d != 0) else np.nan
        mc = []
        for seed in SEEDS:
            y, pa = load_pred(split, a, seed); _, pb = load_pred(split, b, seed)
            n01, n10 = int(((pa == y) & (pb != y)).sum()), int(((pa != y) & (pb == y)).sum())
            mc.append(binomtest(min(n01, n10), n01 + n10).pvalue if n01 + n10 else 1.0)
        y, pa = load_pred(split, a, SEEDS[0]); _, pb = load_pred(split, b, SEEDS[0])
        rs = np.random.default_rng(0); boots = []
        for _ in range(N_BOOTSTRAP):
            i = rs.integers(0, len(y), len(y))
            boots.append(f1_score(y[i], pa[i], average="macro", zero_division=0) -
                         f1_score(y[i], pb[i], average="macro", zero_division=0))
        lo, hi = np.percentile(boots, [2.5, 97.5])
        rows.append({"split": split, "A": a, "B": b, "mean diff (A-B)": d.mean(), "std diff": d.std(ddof=1) if len(d) > 1 else np.nan,
                     "A wins (seeds)": f"{int((d > 0).sum())}/{len(d)}", "paired t p": p_t, "Wilcoxon p": p_w,
                     "McNemar p<0.05 (seeds)": f"{sum(p < 0.05 for p in mc)}/{len(mc)}",
                     "bootstrap 95% CI (seed 0)": f"[{lo:+.4f}, {hi:+.4f}]",
                     "verdict": ("A better" if (d.mean() > 0 and (p_t < 0.05) and lo > 0) else
                                 "B better" if (d.mean() < 0 and (p_t < 0.05) and hi < 0) else "no significant difference")})
stats = pd.DataFrame(rows)
stats.to_csv(RES / "statistical_tests.csv", index=False)
display(stats.round(4))

## 15. Figures

In [ ]:
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
COL = {"gatv2": "#8fa9bf", "transformer": "#c9a66b", "two_stage": "#6b9c8a", "hybrid": "#2e5e8c"}
mods = [m for m in MODELS if m in main.model.unique()]

# (a) headline metrics per split, mean ± std
metrics_plot = ["L3 Macro-F1", "L3-merged Macro-F1", "L1 DR @val-thr"]
fig, axes = plt.subplots(1, len(metrics_plot), figsize=(5.2 * len(metrics_plot), 4.2), sharey=False)
for ax, met in zip(axes, metrics_plot):
    x = np.arange(len(SPLITS_TO_RUN)); w = 0.8 / max(len(mods), 1)
    for j, m in enumerate(mods):
        g = main[main.model == m].groupby("split")[met].agg(["mean", "std"]).reindex(SPLITS_TO_RUN)
        ax.bar(x + j * w - 0.4 + w / 2, g["mean"], w, yerr=g["std"].fillna(0), color=COL.get(m), label=m, capsize=3)
    ax.set_xticks(x); ax.set_xticklabels(SPLITS_TO_RUN); ax.set_title(met)
    lo = np.nanmin(main[met]) if main[met].notna().any() else 0
    ax.set_ylim(max(0, lo - 0.1), 1.0); ax.grid(axis="y", alpha=0.25)
axes[0].legend(frameon=False, fontsize=9)
plt.suptitle(f"Mean ± std over {len(SEEDS)} seeds — DR at the threshold giving {TARGET_FAR:.0%} FAR on validation", y=1.03)
plt.savefig(FIG / "02_models_by_split.png", dpi=160, bbox_inches="tight"); plt.show()

# (b) per-class F1 heatmap (mean over seeds) for each split
pcf = pd.DataFrame(PERCLASS)
pcf = pcf[pcf.model.isin(MODELS)]
for split in SPLITS_TO_RUN:
    M = pcf[pcf.split == split].groupby("model")[CLASSES].mean().reindex(mods).T
    fig, ax = plt.subplots(figsize=(1.4 * len(mods) + 4, 0.42 * N_CLASSES + 1.5))
    ax.imshow(M.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(mods))); ax.set_xticklabels(mods); ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASSES)
    for i in range(N_CLASSES):
        for j in range(len(mods)):
            ax.text(j, i, f"{M.values[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if M.values[i, j] > 0.6 else "black")
    ax.set_title(f"Per-class F1 — {split} split (mean over seeds)")
    plt.savefig(FIG / f"03_per_class_f1__{split}.png", dpi=160, bbox_inches="tight"); plt.show()

# (c) where does the validation loss come from?
vlf = pd.DataFrame(VALLOSS)
vlf = vlf[vlf.model.isin(MODELS) & (vlf.split == SPLITS_TO_RUN[0])].groupby("model")[CLASSES].mean().reindex(mods)
vlf.T.to_csv(RES / "per_class_val_loss_mean.csv")
fig, ax = plt.subplots(figsize=(10, 0.4 * N_CLASSES + 1.5))
yy = np.arange(N_CLASSES); h = 0.8 / max(len(mods), 1)
for j, m in enumerate(mods):
    ax.barh(yy + j * h - 0.4 + h / 2, vlf.loc[m].values, h, color=COL.get(m), label=m)
ax.set_yticks(yy); ax.set_yticklabels(CLASSES); ax.invert_yaxis(); ax.legend(frameon=False)
ax.set_xlabel("mean cross-entropy on validation (best checkpoint)")
ax.set_title(f"Per-class validation loss — {SPLITS_TO_RUN[0]} split")
plt.savefig(FIG / "04_per_class_val_loss.png", dpi=160, bbox_inches="tight"); plt.show()

# (d) confusion matrix of the Hybrid, first seed, every split (row-normalised)
for split in SPLITS_TO_RUN:
    f = PRED / f"{split}__hybrid__seed{SEEDS[0]}.npz"
    if not f.exists():
        continue
    z = np.load(f); cm = confusion_matrix(z["y"], z["pred"], labels=np.arange(N_CLASSES)).astype(float)
    cm /= np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(11, 9)); ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASSES, rotation=75, ha="right", fontsize=8)
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASSES, fontsize=8)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            if cm[i, j] >= 0.01:
                ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                        color="white" if cm[i, j] > 0.6 else "black")
    ax.set_title(f"Hybrid — row-normalised confusion matrix — {split} split"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.savefig(FIG / f"05_confusion_hybrid__{split}.png", dpi=160, bbox_inches="tight"); plt.show()

## 16. Package

In [ ]:
zip_path = shutil.make_archive(str(WORK_DIR.parent / "PROTOCOL_V2_RESULTS"), "zip", root_dir=str(WORK_DIR))
print("RUN COMPLETE ->", zip_path)

## How to read the results
1. **`full_dataset_class_by_day.csv`** — confirms on which days Benign exists. If Benign is on every day, the old "Benign only on day 1" was caused by the old sampling.
2. **Ambiguity** — share of each class whose exact feature vector also occurs with another label. A class with a high share cannot be separated at L3 by *any* model → report it at **L3-merged / L2**.
3. **random vs host_disjoint vs temporal** — a large drop from random → host_disjoint means the model memorises hosts. Check `split_report__host_disjoint.csv` for which classes could really be made host-disjoint.
4. **Probes** — time-only RF high ⇒ timestamp shortcut; structure-only GATv2 high ⇒ graph shape alone reveals the label; big drop without TTL/TCP-window ⇒ host fingerprinting; small drop on rewired graph ⇒ the graph adds little.
5. **statistical_tests.csv** — claim "Hybrid better than X" **only** when the verdict says so on the split you report.
6. **per_class_val_loss** — the classes with the highest validation loss are where the rising val-loss ("overfitting") comes from.